# HydroPredict LSTM training on Colab

This notebook calls the same Python modules as the local project. Upload the ZIP produced
by `scripts/package_colab.py`. Select **Runtime → Change runtime type → GPU** before running.
The laptop does not need a GPU or a live connection to Colab. Checkpoints are saved to your
Google Drive. Full training and validation results are pending until you execute this notebook.

The declared initial experiment uses seven observations (about 30 minutes), a compact
32-unit LSTM encoder/decoder and a 16-unit bottleneck. Thirty epochs maximum, early stopping,
and a separate normal-reference calibration subset are fixed in the repository config.
Do not tune using final test results. Publisher-selected data and limited fault coverage
remain limitations. Start with a fresh hosted runtime for a new project snapshot.


In [ ]:
import hashlib
import json
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly the HydroPredict input ZIP.")
WORK = Path(tempfile.mkdtemp(prefix="hydropredict-"))
archive = WORK / "input.zip"
archive.write_bytes(next(iter(uploaded.values())))


def extract_owned_zip(path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as bundle:
        for entry in bundle.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
        bundle.extractall(destination)


INPUT = WORK / "inputs"
extract_owned_zip(archive, INPUT)
manifest = json.loads((INPUT / "input_manifest.json").read_text())
for name, expected in manifest["files_sha256"].items():
    if hashlib.sha256((INPUT / name).read_bytes()).hexdigest().upper() != expected:
        raise ValueError(f"Input checksum mismatch: {name}")
PROJECT = WORK / "project"
extract_owned_zip(INPUT / "project.zip", PROJECT)
DATA = INPUT / "data"
print("Verified source commit:", manifest["source_commit"])

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT) + "[lstm,analysis]"], check=True
)

In [ ]:
import torch

from hydropredict.models.lstm import Autoencoder, LSTMConfig

if not torch.cuda.is_available():
    raise RuntimeError("No hosted GPU. Select a GPU runtime; full training requires CUDA.")
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
CONFIG = PROJECT / "configs" / "lstm_initial.json"
print(CONFIG.read_text())
config = LSTMConfig(**json.loads(CONFIG.read_text()))
print("Model parameters:", sum(p.numel() for p in Autoencoder(config).parameters()))

## Persistent output and recovery

Connecting Drive below is an interactive action you perform in Colab. Use one run name
for this experiment; changing it creates a separate run. If Colab disconnects, upload the
same input ZIP in a fresh runtime, use the same name and rerun the notebook. The training
command resumes from a matching checkpoint; changed data or configuration is rejected.
Free GPU availability and session duration are not guaranteed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUN_NAME = "hydropredict-lstm-initial-v1"  # Keep fixed when resuming this experiment.
RUN_ROOT = Path("/content/drive/MyDrive/HydroPredict") / RUN_NAME
RUN_ROOT.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable,
    str(PROJECT / "scripts/train_lstm.py"),
    "--data-dir",
    str(DATA),
    "--config",
    str(CONFIG),
    "--output",
    str(RUN_ROOT),
]
if (RUN_ROOT / "bundle").exists():
    print("Bundle exists. Continue to evaluation; use a new name to retrain.")
else:
    if (RUN_ROOT / "checkpoint.pt").exists():
        command.append("--resume")
    subprocess.run(command, cwd=PROJECT, check=True)
print("Persistent run:", RUN_ROOT)

## Separate development evaluation

The model and threshold are frozen before fault timestamps are read. This cell evaluates
only the declared validation interval, using original event coverage, model-score coverage,
lead time and selected normal-reference alert burden. It does not establish operational
false-alert rates or an independent final test result. A CPU smoke bundle cannot be used
as a completed scientific experiment.


In [ ]:
EVALUATION = RUN_ROOT / "validation"
if EVALUATION.exists():
    print("Preserving existing validation results:", EVALUATION)
else:
    subprocess.run(
        [
            sys.executable,
            str(PROJECT / "scripts/evaluate_lstm.py"),
            "--bundle",
            str(RUN_ROOT / "bundle"),
            "--data-dir",
            str(DATA),
            "--output",
            str(EVALUATION),
        ],
        cwd=PROJECT,
        check=True,
    )
report = json.loads((EVALUATION / "evaluation.json").read_text())
for section in ["original_eligible", "adequately_scored", "nominal", "sequence_accounting"]:
    value = report[section]
    print(
        section,
        {k: v for k, v in value.items() if k != "events"} if isinstance(value, dict) else value,
    )

## Export to the laptop

Download the archive below. Extract it into a new folder under local `models/`. Its `bundle/`
contains CPU-loadable weights, scaler, sequence configuration, threshold, version and
checksums. Keep the training and validation reports with it. There is no automatic sync.
Locally install CPU PyTorch as documented, then use `SHPLSTMAutoencoder.load(path_to_bundle)`.
GPU-to-CPU export is checked by the training module. Small floating-point differences
between hardware are allowed within the declared tolerance.


In [ ]:
import shutil

export = shutil.make_archive(str(WORK / "hydropredict-lstm-export"), "zip", root_dir=RUN_ROOT)
files.download(export)
print("Drive remains the persistent copy:", RUN_ROOT)